# Spam SMS classifier, version 2

My first version (`01_tutorial_baseline.ipynb`) followed a tutorial: TF-IDF + logistic regression, 96.6% test accuracy.
This notebook asks: **is 96.6% actually good?** It turns out the answer is no, and fixing that taught me more than building the first model.

In [1]:
import sys, json
import pandas as pd
sys.path.insert(0, "../src")
df = pd.read_csv("../data/spam.csv")
print(df.shape)
df["Category"].value_counts(normalize=True).round(3)

(5572, 2)


Category
ham     0.866
spam    0.134
Name: proportion, dtype: float64

## Problem 1: accuracy hides the class imbalance
Only about 13% of messages are spam. A "model" that says **ham every time** is already about 87% accurate.
So 96.6% accuracy is not as impressive as it sounds. From here on, **spam is the positive class**, and I track:
- **Recall:** what share of spam did we catch?
- **Precision:** when we flag something as spam, how often are we right?

## Problem 2: duplicate messages leak into the test set
The same message (e.g. a common spam blast) can appear many times. If one copy lands in training and another in testing,
the model is tested on text it has already seen.

In [2]:
dupes = df.duplicated(subset="Message").sum()
print(f"{dupes} duplicate messages ({dupes / len(df):.1%} of the data)")
df[df.duplicated(subset="Message", keep=False)].groupby("Message").size().sort_values(ascending=False).head(5)

403 duplicate messages (7.2% of the data)


Message
Sorry, I'll call later                                                                                                                                                                 30
I cant pick the phone right now. Pls send a message                                                                                                                                    12
Ok...                                                                                                                                                                                  10
Okie                                                                                                                                                                                    4
Wen ur lovable bcums angry wid u, dnt take it seriously.. Coz being angry is d most childish n true way of showing deep affection, care n luv!.. kettoda manda... Have nice day da.     4
dtype: int64

## Fixes
1. Remove duplicates **before** splitting.
2. Stratified train/test split, so both sets keep the same spam ratio.
3. Compare models with **5-fold cross-validation** on the training set, then check once on the untouched test set.
4. Try the models from my "future improvements" list: Naive Bayes, a class-balanced logistic regression, and a linear SVM on word + character n-grams.

All of this lives in `src/train.py`:

In [3]:
!cd .. && python src/train.py 2>/dev/null | head -30

5572 messages, 403 duplicates removed, 5169 left, 12.6% spam



5-fold cross-validation (training set):


                                cv_accuracy  cv_spam_precision  cv_spam_recall  cv_spam_f1  cv_spam_f1_std
model                                                                                                     
Always say ham (dummy)                0.874              0.000           0.000       0.000           0.000
Logistic Regression (original)        0.954              0.985           0.644       0.777           0.046
Naive Bayes                           0.984              0.977           0.893       0.933           0.014
Logistic Regression, balanced         0.981              0.944           0.906       0.925           0.014
Linear SVM, words + chars             0.990              0.992           0.929       0.959           0.006



Test set (20%, never used for choosing):
                                accuracy  spam_precision  spam_recall  spam_f1
Always say ham (dummy)             0.873           0.000        0.000    0.000
Logistic Regression (original)     0.957           0.978        0.679    0.802
Naive Bayes                        0.981           0.959        0.885    0.921
Logistic Regression, balanced      0.979           0.929        0.901    0.915
Linear SVM, words + chars          0.985           0.960        0.924    0.942

Best by cross-validated spam F1: Linear SVM, words + chars


Threshold 0.50 -> {'accuracy': 0.9845261121856866, 'spam_precision': 0.952755905511811, 'spam_recall': 0.9236641221374046, 'spam_f1': 0.937984496124031}
Threshold 0.81 -> {'accuracy': 0.9854932301740812, 'spam_precision': 0.9915254237288136, 'spam_recall': 0.8931297709923665, 'spam_f1': 0.9397590361445783}


Saved model, metrics and charts.


In [4]:
m = json.load(open("../reports/metrics.json"))
pd.DataFrame(m["test_set"]).T.mul(100).round(1)

,accuracy,spam_precision,spam_recall,spam_f1
Always say ham (dummy),87.3,0.0,0.0,0.0
Logistic Regression (original),95.7,97.8,67.9,80.2
Naive Bayes,98.1,95.9,88.6,92.1
"Logistic Regression, balanced",97.9,92.9,90.1,91.5
"Linear SVM, words + chars",98.6,96.0,92.4,94.2


**The original model let about a third of spam through** (recall around 68%), even with 95.7% accuracy on the
de-duplicated test set. The linear SVM on words + character n-grams catches about 92% of spam. Character n-grams help with
spammer tricks like `FR33`, `£1000` and `www.` links.

![](../reports/figures/model_comparison.png)

## Problem 3: which mistake is worse?
For a spam filter, hiding a **real** message (a false positive) is worse than letting one spam through.
So instead of the default 0.5 cut-off, I pick the threshold that keeps **precision ≥ 99%** with as much recall as possible.

In [5]:
pd.DataFrame({"threshold 0.5": m["threshold"]["at_0.5"], f"threshold {m['threshold']['value']}": m["threshold"]["tuned"]}).mul(100).round(1)

,threshold 0.5,threshold 0.807
accuracy,98.4,98.6
spam_precision,95.3,99.2
spam_recall,92.4,89.3
spam_f1,93.8,94.0


In [6]:
m["confusion_matrix_tuned"]

{'ham_kept': 902,
 'ham_wrongly_flagged': 1,
 'spam_missed': 14,
 'spam_caught': 117}

![](../reports/figures/precision_recall.png)

![](../reports/figures/confusion_matrix.png)

## What does it still get wrong?

In [7]:
pd.set_option("display.max_colwidth", 120)
pd.read_csv("../reports/error_samples.csv").head(10)

,message,is_spam,spam_probability,predicted
0,"In The Simpsons Movie released in July 2007 name the band that died at the start of the film? A-Green Day, B-Blue Da...",1,0.013,0
1,Would you like to see my XXX pics they are so hot they were nearly banned in the uk!,1,0.049,0
2,ringtoneking 84484,1,0.115,0
3,"Latest News! Police station toilet stolen, cops have nothing to go on!",1,0.157,0
4,Burger King - Wanna play footy at a top stadium? Get 2 Burger King before 1st Sept and go Large or Super with Coca-C...,1,0.204,0
5,Babe: U want me dont u baby! Im nasty and have a thing 4 filthyguys. Fancy a rude time with a sexy bitch. How about ...,1,0.304,0
6,Want explicit SEX in 30 secs? Ring 02073162414 now! Costs 20p/min,1,0.344,0
7,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE MINS. INDIA CUST SERVs SED YES. L8ER GOT MEGA BILL. 3 DONT GIV A SHIT....,1,0.381,0
8,Missed call alert. These numbers called but left no message. 07008009200,1,0.383,0
9,Hi its LUCY Hubby at meetins all day Fri & I will B alone at hotel U fancy cumin over? Pls leave msg 2day 0909972639...,1,0.442,0


## Try it, including a modern Indian scam

In [8]:
!cd .. && python src/predict.py "Congratulations! You won a free iPhone, call 09061701461 now" "are we still meeting at 6?" "URGENT: your account will be blocked, update KYC at bit.ly/xyz" "Your OTP for login is 482910. Do not share it." 

SPAM  (100% spam)  Congratulations! You won a free iPhone, call 09061701461 now
ham   (0% spam)  are we still meeting at 6?
ham   (29% spam)  URGENT: your account will be blocked, update KYC at bit.ly/xyz
ham   (2% spam)  Your OTP for login is 482910. Do not share it.


**Limitation, found by testing:** the KYC scam gets a low spam score. This dataset is from 2011 and mostly UK messages,
so it has never seen today's Indian scams (KYC updates, fake electricity bills, UPI collect requests). That's **data drift**:
a model is only as current as its training data. A next step would be to collect and label recent Indian scam SMS.

## Summary
| | v1 (tutorial) | v2 (this notebook) |
|---|---|---|
| Duplicates | in both train and test | removed before splitting |
| Main metric | accuracy | spam precision and recall |
| Model choice | one model | 4 models, 5-fold cross-validation |
| Decision threshold | 0.5 | tuned for ≥ 99% precision |
| Spam caught | about 68% | about 89%, with 99% precision |